# Building an AI Agent from Scratch: MCP + Gemini

Tools like **Claude Code**, **GPT Codex**, and **Google Jules** can write code, read files, search the web, and take actions on your behalf. But how do they actually work under the hood?

The answer involves three pieces:

1. **An LLM** (the "brain") that reasons about what to do — but it can't actually *do* anything
2. **Tools** (functions exposed by an MCP Server — calculator, file reader, web search, etc.)
3. **An MCP Client** (the orchestrator that sits between you, the LLM, and the tools)

In this notebook, we'll build a working AI agent by wiring together:
- An **MCP Server** that exposes tools
- An **MCP Client** (our Python code right here in Colab!) that discovers tools, talks to the LLM, and executes tool calls
- **Google Gemini** as the LLM that decides *which* tools to call and *when*

By the end, you'll have a mini version of the same architecture that powers Claude Code and friends.

## Architecture Overview

This is the key thing to understand: **the LLM never calls tools directly.** It can only *request* tool calls. The **MCP Client** is the orchestrator that runs the show.

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│    YOU (Google Colab)                                               │
│      │                                                              │
│      │  Your prompt                                                 │
│      ▼                                                              │
│   ┌───────────────────────────────────────────────────────┐         │
│   │              MCP CLIENT  (our Python code)            │         │
│   │                                                       │         │
│   │   This is the orchestrator. It:                       │         │
│   │    1. Connects to MCP Server, discovers tools         │         │
│   │    2. Sends your prompt + tool list ──► Gemini API    │         │
│   │    3. Gets back either:                               │         │
│   │       • text answer ──► done, return to you           │         │
│   │       • tool request ──► execute via MCP Server       │         │
│   │    4. Sends tool result back to Gemini ──► loop       │         │
│   │                                                       │         │
│   │          ┌──────────┐          ┌────────────────┐     │         │
│   │     ┌───►│  Gemini  │     ┌───►│  MCP Server    │     │         │
│   │     │    │  (API)   │     │    │                │     │         │
│   │     │    │ Receives │     │    │  - calculator  │     │         │
│   │     │    │ prompt + │     │    │  - read_file   │     │         │
│   │     │    │ tool list│     │    │  - write_file  │     │         │
│   │     │    │          │     │    │  - list_files  │     │         │
│   │     │    │ Returns: │     │    │  - web_search  │     │         │
│   │     │    │ text or  │     │    │                │     │         │
│   │  prompts │ tl call  │ execute  │  (subprocess)  │     │         │
│   │  + tool  │ request  │  tool    │                │     │         │
│   │  list    └──────────┘  call    └────────────────┘     │         │
│   │     │         │        │              │               │         │
│   │     └─────────┘        └──────────────┘               │         │
│   │      Gemini never                                     │         │
│   │      executes anything.     MCP Server never          │         │
│   │      It only *suggests*     talks to Gemini.          │         │
│   │      tool calls.            It just runs functions.   │         │
│   └───────────────────────────────────────────────────────┘         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### The analogy to real products

| Product | What is the MCP Client? | What is the LLM? | What are the tools? |
|---------|------------------------|-------------------|---------------------|
| **Claude Code** | The CLI application | Claude (API) | File system, git, web search, etc. |
| **GPT Codex** | OpenAI's agent runtime | GPT-4 (API) | Code sandbox, file I/O, shell |
| **Google Jules** | Google's agent runtime | Gemini (API) | Code analysis, file editing |
| **This notebook** | Our `orchestrate()` function | Gemini (API) | Our `mcp_server.py` |

The LLM is always just an API call that returns text or tool-call requests. The client does everything else.

### Why MCP?

Without MCP, every LLM provider would need custom integrations for every tool. MCP provides a **universal standard** — any MCP server works with any MCP client, regardless of which LLM is behind it. Think of it like USB: one protocol, many devices.

---

## Part 0: Setup

We need three packages:
- `mcp` — the MCP protocol SDK (client + server)
- `google-genai` — Google's Gemini SDK
- `nest_asyncio` — lets async code run inside Colab's event loop

In [1]:
!pip install -q "mcp[cli]" google-genai nest_asyncio

In [2]:
import nest_asyncio
nest_asyncio.apply()  # allows asyncio.run() inside Colab's existing event loop

---

## Part 1: Building an MCP Server

An MCP server **exposes tools** that any MCP client can discover and call. Think of it as a menu of capabilities.

We'll create a server with a few simple tools:
- **`calculator`** — evaluates math expressions
- **`list_files`** — lists files in a directory
- **`read_file`** — reads a file's contents
- **`write_file`** — writes content to a file
- **`web_search`** — simulates a web search (returns mock results)

The server is a standalone Python script. We'll write it to disk, and then the MCP client will launch it as a **subprocess** and communicate over **stdio** (standard input/output) — the same transport that VS Code extensions and Language Servers use.

In [3]:
%%writefile mcp_server.py
"""
A simple MCP server that exposes a handful of tools.

This file runs as a subprocess — the MCP client launches it and
communicates over stdin/stdout using the MCP protocol.

IMPORTANT: This server knows nothing about Gemini (or any LLM).
It just exposes tools. Any MCP client can use them.
"""

from mcp.server.fastmcp import FastMCP
import json, os, math

# Create the server with a human-readable name
mcp = FastMCP("demo-tools")


# ── Tool 1: Calculator ────────────────────────────────────────────
@mcp.tool()
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.

    Supports basic arithmetic (+, -, *, /, **) and math functions
    like sqrt(), sin(), cos(), log(), pi, e, etc.

    Args:
        expression: A math expression to evaluate, e.g. '2 + 2' or 'sqrt(144)'
    """
    # We expose only math functions — no builtins for safety
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
    allowed.update({'abs': abs, 'round': round})
    result = eval(expression, {"__builtins__": {}}, allowed)
    return str(result)


# ── Tool 2: List files ────────────────────────────────────────────
@mcp.tool()
def list_files(directory: str = ".") -> str:
    """List files and folders in a directory.

    Args:
        directory: Path to the directory to list. Defaults to current directory.
    """
    try:
        entries = os.listdir(directory)
        return json.dumps(entries, indent=2)
    except FileNotFoundError:
        return f"Error: Directory '{directory}' not found."


# ── Tool 3: Read file ─────────────────────────────────────────────
@mcp.tool()
def read_file(path: str) -> str:
    """Read the contents of a text file.

    Args:
        path: Path to the file to read.
    """
    try:
        with open(path, 'r') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File '{path}' not found."
    except Exception as e:
        return f"Error reading file: {e}"


# ── Tool 4: Write file ────────────────────────────────────────────
@mcp.tool()
def write_file(path: str, content: str) -> str:
    """Write content to a text file. Creates the file if it doesn't exist.

    Args:
        path: Path where the file should be written.
        content: The text content to write.
    """
    try:
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        with open(path, 'w') as f:
            f.write(content)
        return f"Successfully wrote {len(content)} characters to '{path}'."
    except Exception as e:
        return f"Error writing file: {e}"


# ── Tool 5: Web search (simulated) ────────────────────────────────
@mcp.tool()
def web_search(query: str) -> str:
    """Search the web for information on a topic. Returns relevant results.

    Args:
        query: The search query.
    """
    # In a real system this would call a search API.
    # We simulate it here so the notebook works without extra API keys.
    mock_results = [
        {"title": f"Wikipedia: {query}",
         "snippet": f"A comprehensive overview of {query}, covering key concepts and recent developments."},
        {"title": f"{query} — Latest Research (2026)",
         "snippet": f"Recent studies on {query} suggest significant advances in the field."},
    ]
    return json.dumps(mock_results, indent=2)


# ── Run the server ────────────────────────────────────────────────
if __name__ == "__main__":
    mcp.run()  # defaults to stdio transport

Writing mcp_server.py


### What just happened?

We wrote a Python file (`mcp_server.py`) that defines five tools using the `@mcp.tool()` decorator. Notice:

- Each tool is just a **regular Python function** with type hints and a docstring.
- The MCP SDK **automatically generates a JSON schema** from the function signature — any MCP client (and through it, any LLM) will see this schema and know what arguments the tool expects.
- The server communicates over **stdio** — it reads JSON-RPC messages from stdin and writes responses to stdout.

> **Key insight:** The server knows nothing about Gemini, Claude, or any LLM. It just exposes tools via a standard protocol. This is the whole point of MCP.

---

## Part 2: The MCP Client (Discovery)

Now let's play the role of the MCP client — connect to our server and see what tools it offers. The client:
1. **Launches** the server as a subprocess
2. **Discovers** available tools (by calling `list_tools()`)
3. **Calls** tools when asked (by calling `call_tool(name, args)`)

This dynamic discovery is key — the client doesn't need to know what tools exist ahead of time. It asks the server, and the server tells it. When you install a new MCP server in Claude Code, this is exactly what happens.

In [4]:
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def explore_server():
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
    )

    with open(os.devnull, "w") as devnull:
        async with stdio_client(server_params, errlog=devnull) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                tools_response = await session.list_tools()
                print(f"Server exposes {len(tools_response.tools)} tools:\n")

                for tool in tools_response.tools:
                    print(f"  {tool.name}: {tool.description[:80]}")
                    print(f"    Schema: {tool.inputSchema}\n")

                print("-- Test: calling calculator('sqrt(144) + 10') --")
                result = await session.call_tool(
                    "calculator",
                    {"expression": "sqrt(144) + 10"}
                )
                print(f"   Result: {result.content[0].text}")

# Now we call the 'explore_server()' function to get back available tools.
await explore_server()


Server exposes 5 tools:

  calculator: Evaluate a mathematical expression.

    Supports basic arithmetic (+, -, *, /, 
    Schema: {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculatorArguments', 'type': 'object'}

  list_files: List files and folders in a directory.

    Args:
        directory: Path to the
    Schema: {'properties': {'directory': {'default': '.', 'title': 'Directory', 'type': 'string'}}, 'title': 'list_filesArguments', 'type': 'object'}

  read_file: Read the contents of a text file.

    Args:
        path: Path to the file to r
    Schema: {'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'read_fileArguments', 'type': 'object'}

  write_file: Write content to a text file. Creates the file if it doesn't exist.

    Args:
 
    Schema: {'properties': {'path': {'title': 'Path', 'type': 'string'}, 'content': {'title': 'Content', 'type': 'string'}}, 'required

### Key takeaway

The client didn't need to know anything about the server's tools in advance. It **discovered** them dynamically via `list_tools()`. Each tool comes with:
- A **name** (used to call it)
- A **description** (the LLM will read this to decide when to use the tool)
- An **input schema** (tells the LLM what arguments to provide)

Notice that we called `calculator` directly from our client code — no LLM involved. The MCP client can invoke tools on its own. The LLM's role (which we'll add next) is to *decide which tools to call and with what arguments*.

---

## Part 3: Connecting the LLM (Gemini)

Now we add the "brain." Our MCP client will:
1. Get the tool list from the MCP server
2. **Translate** those tool descriptions into Gemini's format (a small adapter)
3. Send the user's prompt + tool descriptions to Gemini
4. If Gemini says "call this tool" → the client executes it via MCP and sends the result back
5. Repeat until Gemini returns a text answer

Gemini **never executes anything.** It only *suggests* tool calls. Our client code does the actual work.

In [ ]:
# ── Set up Gemini ─────────────────────────────────────────────────
#
# You can setup programmatic access to Google Gemini via the Generative Language API - they provide $300 in credits for free.
# See here: https://cloud.google.com/free.
# Once you setup your Google Cloud project, you can go to APIs, enable the Generative Language API, and create an API key under 'Credentials'.

from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = ''
except Exception:
    api_key = None

if api_key:
    gemini_client = genai.Client(api_key=api_key)
else:
    gemini_client = genai.Client()  # uses default credentials

GEMINI_MODEL = "gemini-3.1-pro-preview"

print(f"Gemini client ready (model: {GEMINI_MODEL})")

Gemini client ready (model: gemini-3.1-pro-preview)


### The adapter: MCP tool schemas to Gemini format

MCP and Gemini describe tools in slightly different JSON formats. We need a small adapter to translate. In production agents, these adapters exist for each LLM provider — same tools, different wire formats.

In [6]:
def mcp_tools_to_gemini(mcp_tools) -> types.Tool:
    """
    Convert a list of MCP tool definitions into a Gemini Tool object.

    MCP tools have an `inputSchema` (JSON Schema).  Gemini expects
    the same thing via `parameters_json_schema`.
    """
    declarations = []
    for tool in mcp_tools:
        declarations.append(
            types.FunctionDeclaration(
                name=tool.name,
                description=tool.description,
                parameters_json_schema=tool.inputSchema,
            )
        )
    return types.Tool(function_declarations=declarations)


print("Adapter function defined.")

Adapter function defined.


---

## Part 4: The Agentic Loop

This is the heart of the system — and it lives entirely in our MCP client code (not in Gemini!).

```
while True:
    1. Client sends conversation history + tool descriptions → Gemini API
    2. Gemini returns either:
       (a) A TEXT response       → client returns it to the user. Done.
       (b) A TOOL CALL request   → client executes it via MCP server,
                                   appends the result, loops back to 1.
```

The LLM can chain multiple tool calls before producing a final answer — this is what makes it an **agent** rather than a simple chatbot. But the LLM is never "in control" — our client code is always the one deciding to actually execute the tool call and loop back.

In [7]:
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

import traceback

async def orchestrate(user_query: str, verbose: bool = True):
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
    )

    try:
        with open("/tmp/mcp_server_stderr.log", "w") as errlog:
            async with stdio_client(server_params, errlog=errlog) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()

                    tools_response = await session.list_tools()
                    gemini_tool = mcp_tools_to_gemini(tools_response.tools)

                    if verbose:
                        tool_names = [t.name for t in tools_response.tools]
                        print(f"[Client] Connected to MCP server. Tools: {tool_names}\n")

                    config = types.GenerateContentConfig(
                        tools=[gemini_tool],
                        automatic_function_calling=types.AutomaticFunctionCallingConfig(
                            disable=True
                        ),
                    )

                    messages = [
                        types.Content(
                            role="user",
                            parts=[types.Part.from_text(text=user_query)],
                        )
                    ]

                    if verbose:
                        print(f"[User]   {user_query}\n")

                    step = 0
                    while True:
                        step += 1

                        response = gemini_client.models.generate_content(
                            model=GEMINI_MODEL,
                            contents=messages,
                            config=config,
                        )

                        if not response.candidates:
                            raise RuntimeError(f"No candidates returned. Full response: {response}")

                        candidate = response.candidates[0]
                        if not candidate.content or not candidate.content.parts:
                            raise RuntimeError(f"Candidate had no content/parts. Full response: {response}")

                        model_content = candidate.content

                        function_calls = [
                            part for part in model_content.parts
                            if getattr(part, "function_call", None)
                        ]

                        if not function_calls:
                            text = getattr(response, "text", None)
                            if verbose:
                                print(f"[Gemini] Final answer (step {step}):\n{text}")
                            else:
                                print(text)
                            return text

                        messages.append(model_content)

                        function_response_parts = []
                        for part in function_calls:
                            fc = part.function_call
                            args = dict(fc.args) if fc.args else {}

                            if verbose:
                                print(f"[Gemini] Step {step} — requests tool: {fc.name}({args})")

                            result = await session.call_tool(fc.name, args)
                            result_text = result.content[0].text if result.content else ""

                            if verbose:
                                preview = result_text[:200] + ("..." if len(result_text) > 200 else "")
                                print(f"[Client] Tool result: {preview}\n")

                            function_response_parts.append(
                                types.Part.from_function_response(
                                    name=fc.name,
                                    response={"result": result_text},
                                )
                            )

                        messages.append(
                            types.Content(role="tool", parts=function_response_parts)
                        )

    except* Exception as eg:
        print("Unhandled exception group:")
        for i, exc in enumerate(eg.exceptions, 1):
            print(f"\n[{i}] {type(exc).__name__}: {exc}")
            traceback.print_exception(type(exc), exc, exc.__traceback__)

### Read the log prefixes carefully

When you run the examples below, notice the prefixes:
- **`[User]`** — your prompt
- **`[Client]`** — our MCP client code (the orchestrator)
- **`[Gemini]`** — what the LLM said or requested

Gemini only ever *requests* tool calls. The `[Client]` is the one that actually executes them and decides to loop back.

---

## Part 5: Let's Try It!

Let's ask our agent some questions that require it to use different tools.

### Example 1: A math question

The agent should recognize it needs the `calculator` tool.

In [8]:
await orchestrate("What is the square root of 1_764, plus 58 divided by 2?")

[Client] Connected to MCP server. Tools: ['calculator', 'list_files', 'read_file', 'write_file', 'web_search']

[User]   What is the square root of 1_764, plus 58 divided by 2?

Unhandled exception group:

[1] ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)


  + Exception Group Traceback (most recent call last):
  |   File "/usr/local/lib/python3.12/dist-packages/mcp/client/stdio/__init__.py", line 189, in stdio_client
  |     yield read_stream, write_stream
  |   File "/tmp/ipykernel_327/254158614.py", line 16, in orchestrate
  |     async with ClientSession(read, write) as session:
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.12/dist-packages/mcp/shared/session.py", line 238, in __aexit__
  |     return await self._task_group.__aexit__(exc_type, exc_val, exc_tb)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py", line 783, in __aexit__
  |     raise BaseExceptionGroup(
  | ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   File "/tmp/ipykernel_327/254158614.py", line 47, in orchestrate
    |     

### Example 2: File operations

This requires **two sequential tool calls** — write, then read. Watch how the client loops twice.

In [ ]:
await orchestrate(
    "Create a file called 'hello.txt' containing a short poem about AI, "
    "then read the file back and tell me what it says."
)

[Client] Connected to MCP server. Tools: ['calculator', 'list_files', 'read_file', 'write_file', 'web_search']

[User]   Create a file called 'hello.txt' containing a short poem about AI, then read the file back and tell me what it says.

[Gemini] Step 1 — requests tool: write_file({'content': 'Lines of code that spark and glow,\nLearning things we do not know.\nSilicon mind and endless streams,\nAwakening to electric dreams.', 'path': 'hello.txt'})
[Client] Tool result: Successfully wrote 130 characters to 'hello.txt'.

[Gemini] Step 2 — requests tool: read_file({'path': 'hello.txt'})
[Client] Tool result: Lines of code that spark and glow,
Learning things we do not know.
Silicon mind and endless streams,
Awakening to electric dreams.

[Gemini] Final answer (step 3):
I have created the file `hello.txt` and written the poem to it. 

Here is what it says when read back from the file:

*Lines of code that spark and glow,*
*Learning things we do not know.*
*Silicon mind and endless stream

'I have created the file `hello.txt` and written the poem to it. \n\nHere is what it says when read back from the file:\n\n*Lines of code that spark and glow,*\n*Learning things we do not know.*\n*Silicon mind and endless streams,*\n*Awakening to electric dreams.*'

### Example 3: Multi-tool reasoning

This requires the agent to combine information from multiple tools.

In [ ]:
await orchestrate(
    "Search the web for 'transformer architecture', then calculate how many "
    "parameters a model with 96 layers, 12_288 hidden dimensions, and 96 attention "
    "heads would have (approximately: 12 * hidden_dim^2 * num_layers). "
    "Finally, write a short summary to 'research_notes.txt'."
)

### Example 4: Try your own!

Modify the query below. Available tools: `calculator`, `list_files`, `read_file`, `write_file`, `web_search`.

In [ ]:
await orchestrate("YOUR QUESTION HERE")

---

## Part 6: What's Really Going On?

### The step-by-step flow

```
User: "What is sqrt(144) + 10?"
         │
         ▼
   MCP CLIENT (our code)
         │
         │  Sends to Gemini API:
         │   - user prompt
         │   - tool descriptions (from MCP server)
         │
         ▼
   GEMINI API responds:
         │  "I'd like to call calculator(expression='sqrt(144) + 10')"
         │
         │  (Note: Gemini did NOT call anything.
         │   It returned a structured request.)
         │
         ▼
   MCP CLIENT receives the request and decides to execute it.
         │
         │  Calls: session.call_tool('calculator', ...)
         │  MCP Server runs the function, returns '22.0'
         │
         ▼
   MCP CLIENT sends the result back to Gemini:
         │  "The calculator returned 22.0"
         │
         ▼
   GEMINI API responds with text:
         │  "The square root of 144 is 12, plus 10 equals 22."
         │
         ▼
   MCP CLIENT returns the text to the user.
```

### What makes this an "agent"?

A traditional chatbot just generates text. An **agent** architecture adds:

1. **Reasoning** — the LLM decides what tools to use (but doesn't execute them)
2. **Action** — the client executes tool calls via MCP
3. **Observation** — the client feeds results back to the LLM
4. **Iteration** — the client loops until the LLM produces a final answer

This is sometimes called the **ReAct** pattern (Reason + Act). Our `orchestrate()` function implements exactly this loop.

### Who is really in control?

The **MCP client** (our code) is always in control. It decides:
- Whether to actually execute a tool call the LLM requested
- Whether to keep looping or stop
- What information to send back to the LLM

This is why products like Claude Code can have permission systems ("allow/deny this tool call") — the client can inspect what the LLM wants to do and ask the user before executing it.

---

## Part 7: Things You Can Try!

### Exercise 1: Add a new tool
Add a `get_weather(city: str)` tool to `mcp_server.py` that returns simulated weather data. Restart the agent and ask it about the weather.

### Exercise 2: Add a permission system
Modify `orchestrate()` so that before executing any tool call, it prints what Gemini wants to do and asks the user to type 'y' to approve. This is how Claude Code's permission system works.

### Exercise 3: System prompts
Modify `orchestrate()` to accept a `system_prompt` parameter that tells Gemini how to behave (e.g., "You are a cautious research assistant. Always verify information using multiple tools before answering."). How does this change behavior?

### Exercise 4: Conversation memory
Right now, each call to `orchestrate()` starts fresh. Modify it to maintain conversation history across multiple calls, so the agent remembers previous interactions.

### Exercise 5 (Advanced): Real web search
Replace the simulated `web_search` tool with a real one using an API like [SerpAPI](https://serpapi.com/) or [Tavily](https://tavily.com/). How does the agent's behavior change with real search results?

---

## Summary

| Component | Role | What we used |
|-----------|------|--------------|
| **MCP Server** | Exposes tools via a standard protocol. Knows nothing about the LLM. | `mcp.server.fastmcp.FastMCP` |
| **MCP Client** | The orchestrator. Discovers tools, talks to the LLM, executes tool calls, runs the loop. | `mcp.ClientSession` + `stdio_client` + our `orchestrate()` code |
| **LLM (Gemini)** | The "brain." Receives prompts + tool descriptions. Returns text or tool-call *requests*. Never executes anything. | `google-genai` SDK |

The **agentic loop** (in the client) ties them together: send to LLM → if tool request, execute via MCP → send result back → repeat.

This is the same fundamental architecture behind Claude Code, GPT Codex, Google Jules, and other AI agents. The client is always the orchestrator. The LLM is always just an API that suggests actions. The MCP server is always just a tool provider.